# ITSN1 Gene Burden Analysis in the UK Biobank

- Author: Sheila Yeboah
- Project: ITSN1 Burden
- Date Created: 11/12/25
- Date Updated: 3/11/26
- Data: UK Biobank 500k DRAGEN WGS Latest

## Imports

In [1]:
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import math
import sys
import requests
import subprocess
#import statsmodels.api as sm
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from datetime import date, datetime
import pathlib
#from plotnine import *
import pyspark
import dxdata
import dxpy


## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
    # Repurposed code from stackoverflow here: https://stackoverflow.com/questions/40428931/package-for-listing-version-of-packages-used-in-a-jupyter-notebook

## Import packages 
import pkg_resources
import types
from datetime import date
today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package, not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different imported names vs. system/pip names
        # Unfortunately, there is no systematic way to get pip names from a package's imported name. You'll have to add exceptions to this list manually!
        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

# The only way I found to get the version of the root package from only the name of the package is to cross-check the names of installed packages vs. imported packages
requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name!="pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

PACKAGE VERSIONS (04-JAN-2026)
	dxdata==0.47.0
	dxpy==0.396.0
	matplotlib==3.9.2
	numpy==1.26.4
	pandas==2.2.3
	pyspark==3.5.2
	requests==2.32.3
	scipy==1.15.3
	seaborn==0.12.2


/tmp/ipykernel_165/1700372142.py:28: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


## Initialize Spark Session

In [2]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

## Initialize Variables

In [2]:
GENE = "ITSN1_WGS"
gene_names = ["ITSN1"]
OUTPUT_DIR = f"/results/11625_ITSN1_WGS_Pan_Ancestry"
DATA_DIR = "/opt/notebooks/data"
ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "FIN", "MDE", "SAS"]

## Initialize Helper Function

In [4]:
def fetch_gene_info_ensembl(gene_names, species="human", genome_version="GRCh38"):
    gene_info_dict = {}
    server = "https://rest.ensembl.org"
    
    for gene_name in gene_names:
        endpoint = f"/lookup/symbol/{species}/{gene_name}"
        headers = {"Content-Type": "application/json"}

        response = requests.get(server + endpoint, headers=headers, params={"expand": "1"})
        if not response.ok:
            print(f"Fetching failed for {gene_name}")
            continue

        data = response.json()
        gene_info = {
            "gene_name": data.get("display_name", gene_name),
            "chromosome": f"chr{data['seq_region_name']}",
            "start": int(data["start"]),
            "end": int(data["end"]),
            "genome_version": genome_version
        }

        gene_info_dict[gene_name] = gene_info

    return gene_info_dict

## Install RV tests

In [6]:
%%capture
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /opt/notebooks/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /opt/notebooks/tools/rvtests
cd /opt/notebooks/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
# chmod to make sure you have permission to run the program
! chmod 777 /opt/notebooks/tools/rvtests/executable/rvtest

## Fetch participants from each phenotype, control and PD

### Get cases

In [5]:
# pull participant information using Spark
dispensed_dataset_id = dxpy.find_one_data_object(typename="Dataset", name="app*.dataset", folder="/", name_mode="glob")["id"]
dataset = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset["participant"]


In [6]:
# Filter by appropriate fields

field_names = [
    "eid", 
    "p31", 
    "p34", 
    "p21022", 
    "p42032",
    "p40000_i0",
]
df_cases = participant.retrieve_fields(names=field_names, coding_values="replace", engine=dxdata.connect())
df_cases = df_cases.toPandas()

In [7]:
# Change columns to readable names
df_cases = df_cases.rename(columns={
    "eid":"ID",
    "p31":"GENETIC_SEX", 
    "p34":"BIRTH_YEAR", 
    "p21022":"AGE_OF_RECRUIT",
    "p42032":"PD_DATE",
    "p40000_i0":"DATE_OF_DEATH",
})

In [9]:
# convert dnanexus ids to numeric
df_cases["ID"] = pd.to_numeric(df_cases["ID"])

In [ ]:
# get pcs 
! dx download /data/ukbb_imputed_genotypes_proj_pca.txt --overwrite
pcs = pd.read_csv("ukbb_imputed_genotypes_proj_pca.txt", sep="\t")


In [ ]:
pcs = pcs[["IID", "PC1", "PC2", "PC3", "PC4", "PC5"]]
pcs.rename({"IID":"ID"}, inplace=True, axis=1)

In [ ]:
! dx download /data/ukbb_imputed_genotypes_umap_linearsvc_predicted_labels.txt --overwrite
ancestry = pd.read_csv("ukbb_imputed_genotypes_umap_linearsvc_predicted_labels.txt", sep="\t")

In [14]:
ancestry = ancestry[["IID", "label"]]
ancestry.rename({"IID":"ID", "label":"ANCESTRY"}, inplace=True, axis=1)

In [16]:
df_cases = df_cases.merge(pcs, on = "ID")

In [17]:
df_cases = df_cases.merge(ancestry, on = "ID")

### Find participants with PD

In [18]:
df_cases = df_cases[~df_cases[f"PD_DATE"].isna()]
df_cases = df_cases[[
    "ID", 
    "GENETIC_SEX", 
    "BIRTH_YEAR", 
    "AGE_OF_RECRUIT", 
    "PD_DATE", 
    "PC1", 
    "PC2", 
    "PC3", 
    "PC4", 
    "PC5", 
    "DATE_OF_DEATH",
    "ANCESTRY",
]]

## Get Controls

### Retrieve field names of interest for each participant

In [45]:
## Date G20 first reported (parkinson's disease),
## Date G21 first reported (secondary parkinsonism),
## Date G22 first reported (parkinsonism in diseases classified elsewhere),
## Date G23 first reported (other degenerative diseases of basal ganglia),
## Date G24 first reported (dystonia),
## Date G25 first reported (other extrapyramidal and movement disorders),
## Date of all cause parkinsonism report,
## Date of parkinson's disease report,
## Genetic ethnic grouping,
## Age at recruitment,
## Townsend deprivation index at recruitment,
## Sex,
## Genetic Principal components | Array 1,
## Genetic Principal components | Array 2,
## Genetic Principal components | Array 3,
## Genetic Principal components | Array 4,
## Genetic Principal components | Array 5,
## birth month,
## birth year
## death date

field_names = ['eid', 'p131022', 'p131024', 'p131026', 'p131028', 'p131030', 'p131032', 'p42030', 'p42032',
               'p22006', 'p21022', 'p22189', 'p31', 'p22009_a1',
              'p22009_a2', 'p22009_a3', 'p22009_a4', 'p22009_a5', 'p22009_a6','p22009_a7','p22009_a8','p22009_a9','p22009_a10', 'p52', 'p34', 'p40007_i0', 'p40007_i1', 'p40000_i0']
df_control = participant.retrieve_fields(names=field_names, coding_values="replace", engine=dxdata.connect())
df_control = df_control.toPandas()

### Remove participants with any of the listed conditions

In [48]:

# Keep only rows where all listed columns are null
## Date G20 first reported (parkinson's disease),
## Date G21 first reported (secondary parkinsonism),
## Date G22 first reported (parkinsonism in diseases classified elsewhere),
## Date G23 first reported (other degenerative diseases of basal ganglia),
## Date G24 first reported (dystonia),
## Date G25 first reported (other extrapyramidal and movement disorders),
## Date of all cause parkinsonism report,
## Date of parkinson's disease report,
cols_to_check = ['p131022', 'p131024', 'p131026', 'p131028', 'p131030', 'p131032', 'p42030', 'p42032']

df_control = df_control[df_control[cols_to_check].isnull().all(axis=1)]

### Remove participants below the defined age threshold

In [50]:
df_control = df_control[df_control["p21022"] >= 60]

### Rename columns

In [51]:
df_control = df_control[["eid", "p21022", "p31", "p34", "p40000_i0"]]
df_control.rename(columns={
    "eid":"ID",
    "p21022":"AGE_OF_RECRUIT", 
    "p31":"GENETIC_SEX",
    "p34":"BIRTH_YEAR", 
    "p40000_i0":"DATE_OF_DEATH",
}, inplace=True)
df_control["ID"] = pd.to_numeric(df_control["ID"])
df_control.info()

<class 'pandas.core.frame.DataFrame'>

Index: 209719 entries, 0 to 502130

Data columns (total 5 columns):

 #   Column          Non-Null Count   Dtype 

---  ------          --------------   ----- 

 0   ID              209719 non-null  int64 

 1   AGE_OF_RECRUIT  209719 non-null  int64 

 2   GENETIC_SEX     209719 non-null  object

 3   BIRTH_YEAR      209719 non-null  int64 

 4   DATE_OF_DEATH   29398 non-null   object

dtypes: int64(3), object(2)

memory usage: 9.6+ MB


In [52]:
df_control = df_control.merge(pcs, on = "ID")

In [53]:
df_control = df_control.merge(ancestry, on = "ID")

In [55]:
# drop any NAs in the PCs
df_cases.dropna(subset=["PC1","PC2","PC3","PC4","PC5"], inplace=True)
df_control.dropna(subset=["PC1","PC2","PC3","PC4","PC5"], inplace=True)

In [56]:
# get ids for each phenotype
ids_control = df_control["ID"].tolist()
ids_cases = df_cases["ID"].tolist()

## Remove related individuals

In [ ]:
# Fetch relatedness data
! dx download "Bulk/Genotype Results/Genotype calls/ukb_rel.dat" -f --overwrite
df_full_related = pd.read_csv("ukb_rel.dat", sep = " ")
df_full_related = df_full_related[df_full_related["Kinship"] > 0.0884]

In [58]:
# keep rows with both participants
df_related_cohort = df_full_related.loc[df_full_related["ID1"].isin(ids_cases + ids_control) & df_full_related["ID2"].isin(ids_cases + ids_control)]
df_related_cohort = df_related_cohort.reset_index(drop=True)

In [59]:
# maximize the number of cases included
df_flipped = df_related_cohort[df_related_cohort["ID1"].isin(ids_control) & df_related_cohort["ID2"].isin(ids_cases)].copy()
df_related_cohort = df_related_cohort[~(df_related_cohort["ID1"].isin(ids_control) & df_related_cohort["ID2"].isin(ids_cases))]
df_flipped.rename(columns={"ID1":"ID2", "ID2":"ID1"}, inplace=True)
df_related_cohort = pd.concat([df_related_cohort, df_flipped])

In [60]:
# get set of participants to remove
ids_to_remove = set(df_related_cohort["ID2"])
print(f"Removing {len(ids_to_remove)} participants")

Removing 7151 participants


In [61]:
# filter ID lists accordingly
ids_cases = [iid for iid in ids_cases if iid not in ids_to_remove]
ids_control = [iid for iid in ids_control if iid not in ids_to_remove]
ids_combined = ids_cases + ids_control

In [62]:
print(len(ids_cases))
print(len(ids_control))
print(len([iid for iid in ids_control if iid in ids_cases]))

4252

196573

0


## Save ids of each participant into text file

In [63]:
with open("cases_ids_pre_VCF.txt", "w") as file:
    for iid in ids_cases:
        file.write(f"{iid}\n")

In [64]:
with open("control_ids_pre_VCF.txt", "w") as file:
    for iid in ids_control:
        file.write(f"{iid}\n")

In [65]:
with open("ids_pre_VCF.txt", "w") as file:
    for iid in ids_combined:
        file.write(f"{iid}\n")

## Only include participants with WGS data

In [ ]:
%%bash

# any random pvcf works here, don't worry about chromosome matching
# pull ids from pvcf
dx run swiss-army-knife \
-iin="chr1/ukb24310_c1_b7761_v1.vcf.gz" \
-iin="chr1/ukb24310_c1_b7761_v1.vcf.gz.tbi" \
-icmd="bcftools query -l ukb24310_c1_b7761_v1.vcf.gz > pvcf_full_ids.txt" \
--instance-type mem1_hdd1_v2_x16 \
--brief \
--destination "${projectid}:/data"

In [ ]:
! dx download /data/pvcf_full_ids.txt --overwrite
! grep -Fwf pvcf_full_ids.txt ids_pre_VCF.txt > filtered_sample_ids.txt
! grep -Fwf pvcf_full_ids.txt cases_ids_pre_VCF.txt > filtered_cases_ids.txt
! grep -Fwf pvcf_full_ids.txt control_ids_pre_VCF.txt > filtered_control_ids.txt

In [67]:
with open("filtered_cases_ids.txt", "r") as file:
    ids_cases = [int(line.strip()) for line in file]
with open("filtered_control_ids.txt", "r") as file:
    ids_control = [int(line.strip()) for line in file]

## Get participant ids

In [68]:
# Get participant IDs for each phenotype
df_cases = df_cases[df_cases["ID"].isin(ids_cases)]
df_control = df_control[df_control["ID"].isin(ids_control)]

In [69]:
print(f"Number of Cases:                 {len(ids_cases)}")
print(f"Number of Control participants:  {len(ids_control)}")

Number of Cases:                 4225

Number of Control participants:  195437


In [ ]:
! dx upload filtered_sample_ids.txt --path {OUTPUT_DIR}/sample_ids.txt
! dx upload filtered_cases_ids.txt --path {OUTPUT_DIR}/cases_ids.txt
! dx upload filtered_control_ids.txt --path {OUTPUT_DIR}/control_ids.txt

## Generate pheno and sex files

In [71]:
# code males as 1 and females as 2 
cases_genetic_sex = [1 if df_cases["GENETIC_SEX"][df_cases["ID"] == int(iid)].values[0] == "Male" else 2 for iid in ids_cases]
control_genetic_sex = [1 if df_control["GENETIC_SEX"][df_control["ID"] == int(iid)].values[0] == "Male" else 2 for iid in ids_control]
genetic_sex = cases_genetic_sex + control_genetic_sex

pheno = [2] * len(ids_cases) + [1] * len(ids_control)


In [72]:
with open('sex.txt', 'w') as f:
    for line in zip(ids_combined, genetic_sex):
        f.write(f"{line[0]}\t{line[0]}\t{line[1]}\n")
        
with open('pheno.txt', 'w') as f:
    for line in zip(ids_combined, pheno):
        f.write(f"{line[0]}\t{line[0]}\t{line[1]}\n")

In [ ]:
# upload files
! dx upload sex.txt --path {OUTPUT_DIR}/sex.txt
! dx upload pheno.txt --path {OUTPUT_DIR}/pheno.txt

### Save and print cohort statistics

In [76]:
df_control.to_csv("control_age60cutoff.txt",sep ="\t", header=True, index=False)
df_cases.to_csv("cases_age60cutoff.txt",sep ="\t", header=True, index=False)

In [ ]:
! dx upload control_age60cutoff.txt --path {OUTPUT_DIR}/control_age60cutoff.txt
! dx upload cases_age60cutoff.txt --path {OUTPUT_DIR}/cases_age60cutoff.txt

In [79]:
print(df_control["GENETIC_SEX"].value_counts())
print(df_cases["GENETIC_SEX"].value_counts())
print("\n")

print(f'{df_control[df_control["GENETIC_SEX"] == "Male"]["AGE_OF_RECRUIT"].mean()} +/- {df_control[df_control["GENETIC_SEX"] == "Male"]["AGE_OF_RECRUIT"].std()}')
print(f'{df_cases[df_cases["GENETIC_SEX"] == "Male"]["AGE_OF_RECRUIT"].mean()} +/- {df_cases[df_cases["GENETIC_SEX"] == "Male"]["AGE_OF_RECRUIT"].std()}')
print(f'{df_control[df_control["GENETIC_SEX"] == "Female"]["AGE_OF_RECRUIT"].mean()} +/- {df_control[df_control["GENETIC_SEX"] == "Female"]["AGE_OF_RECRUIT"].std()}')
print(f'{df_cases[df_cases["GENETIC_SEX"] == "Female"]["AGE_OF_RECRUIT"].mean()} +/- {df_cases[df_cases["GENETIC_SEX"] == "Female"]["AGE_OF_RECRUIT"].std()}')
print("\n")


GENETIC_SEX

Female    102794

Male       92643

Name: count, dtype: int64

GENETIC_SEX

Male      2650

Female    1575

Name: count, dtype: int64





64.22341677190937 +/- 2.859175624809177

63.04716981132076 +/- 5.257258864794013

64.01008813743992 +/- 2.846612535669247

62.516825396825396 +/- 5.385359708619513






## Fetch pVCF chunks for ITSN1

In [80]:
gene_info = fetch_gene_info_ensembl(gene_names)
print(gene_info)
for gene_name in gene_names:
    start = gene_info[gene_name]["start"] // 20000
    end = fetch_gene_info_ensembl(gene_names)[gene_name]["end"] // 20000 + 1
    print(f"Chromosome:    {fetch_gene_info_ensembl(gene_names)[gene_name]['chromosome']}")
    print(f"Start b-val:   {start}")
    print(f"End b-val:     {end}")

{'ITSN1': {'gene_name': 'ITSN1', 'chromosome': 'chr21', 'start': 33642400, 'end': 33899861, 'genome_version': 'GRCh38'}}

Chromosome:    chr21

Start b-val:   1682

End b-val:     1695


In [ ]:
%%bash -s $OUTPUT_DIR
# Pull vcfs from bulk data, filter with sample ids
for b_val in {1682..1695};
do
    dx run swiss-army-knife \
    -iin="chr21/ukb24310_c21_b${b_val}_v1.vcf.gz" \
    -iin="chr21/ukb24310_c21_b${b_val}_v1.vcf.gz.tbi" \
    -iin="${1}/sample_ids.txt" \
    -icmd="bcftools view -O z -S sample_ids.txt --force-samples ukb24310_c21_b${b_val}_v1.vcf.gz -o ITSN1_b${b_val}_age60cutoff.vcf.gz" \
    --instance-type mem2_ssd1_v2_x32 \
    --brief \
    --destination "${projectid}:${1}/01_pvcf_chunks"
done

## Combine pVCFs into one file

In [ ]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/01_pvcf_chunks/ITSN1_b1682_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1683_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1684_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1685_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1686_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1687_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1688_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1689_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1690_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1691_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1692_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1693_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1694_age60cutoff.vcf.gz" \
-iin="${1}/01_pvcf_chunks/ITSN1_b1695_age60cutoff.vcf.gz" \
-icmd="bcftools concat -O z ITSN1_b1682_age60cutoff.vcf.gz ITSN1_b1683_age60cutoff.vcf.gz ITSN1_b1684_age60cutoff.vcf.gz ITSN1_b1685_age60cutoff.vcf.gz ITSN1_b1686_age60cutoff.vcf.gz ITSN1_b1687_age60cutoff.vcf.gz ITSN1_b1688_age60cutoff.vcf.gz ITSN1_b1689_age60cutoff.vcf.gz ITSN1_b1690_age60cutoff.vcf.gz ITSN1_b1691_age60cutoff.vcf.gz ITSN1_b1692_age60cutoff.vcf.gz ITSN1_b1693_age60cutoff.vcf.gz ITSN1_b1694_age60cutoff.vcf.gz ITSN1_b1695_age60cutoff.vcf.gz -o ITSN1_age60cutoff.vcf.gz" \
--instance-type mem2_ssd1_v2_x64 \
--brief \
--destination "${projectid}:${1}/02_pvcf_genes"

### Split multiallelic sites into biallelic records

In [ ]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/02_pvcf_genes/ITSN1_age60cutoff.vcf.gz" \
-icmd="bcftools norm -m-both -o biallelic_age60cutoff.vcf ITSN1_age60cutoff.vcf.gz" \
--instance-type mem2_ssd1_v2_x64 \
--brief \
--destination "${projectid}:${1}/03_normalized"

### Left align and normalize

In [ ]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/03_normalized/biallelic_age60cutoff.vcf" \
-iin="/data/hg38.fa" \
-icmd="bcftools norm -f hg38.fa -o normalized_age60cutoff.vcf biallelic_age60cutoff.vcf" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--destination "${projectid}:${1}/03_normalized/"

In [ ]:
%%bash -s $OUTPUT_DIR

### remove sites with MAC <1

dx run swiss-army-knife \
-iin="${1}/03_normalized/normalized_age60cutoff.vcf" \
-iin="/data/hg38.fa" \
-icmd="bcftools view -i 'MAC>=1' normalized_age60cutoff.vcf -O v -o normalized_mac1_age60cutoff.vcf" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--destination "${projectid}:${1}/03_normalized/"


### Make a file with updated variant ids

In [ ]:
import subprocess
# Create file with updated variant ids in chr:r:a format
projectid = "project"
plink_cmd = (
    f"plink2 --vcf normalized_mac1_age60cutoff.vcf "
    f"--set-all-var-ids 'chr@:#:\$r:\$a' "
    f"--new-id-max-allele-len 999 "
    f"--double-id "
    f"--make-pgen "
    f"--pheno-name PHENO "
    f"--out normalized_mac1_age60cutoff_updateids"
)

# Construct the full dx run command
dx_cmd = (
    f'dx run swiss-army-knife '
    f'-iin="{OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff.vcf" '
    f'-icmd="{plink_cmd}" '
    f'--brief '
    f'--instance-type mem2_ssd1_v2_x64 '
    f'--destination "{projectid}:{OUTPUT_DIR}/03_normalized/"'
)

# Print for verification
print("Running command:\n", dx_cmd)

# Run the command
subprocess.run(dx_cmd, shell=True, check=True)

In [ ]:
# convert it to vcf

# Create file with updated variant ids in chr:r:a format
projectid = "project"
plink_cmd = (
    f"plink2 --pfile normalized_mac1_age60cutoff_updateids "
    f"--export vcf "
    f"--out normalized_mac1_age60cutoff_updateids"
)

# Construct the full dx run command
dx_cmd = (
    f'dx run swiss-army-knife '
    f'-iin="{OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.pvar" '
    f'-iin="{OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.psam" '
    f'-iin="{OUTPUT_DIR}/03_normalized/normalized_mac1_age60cutoff_updateids.pgen" '
    f'-icmd="{plink_cmd}" '
    f'--brief '
    f'--instance-type mem2_ssd1_v2_x64 '
    f'--destination "{projectid}:{OUTPUT_DIR}/03_normalized/"'
)

# Print for verification
print("Running command:\n", dx_cmd)

# Run the command
subprocess.run(dx_cmd, shell=True, check=True)

### Remove participant genotype info from VCF for annotation

In [3]:
%%bash -s $OUTPUT_DIR

dx run swiss-army-knife \
-iin="${1}/03_normalized/normalized_mac1_age60cutoff_updateids.vcf" \
-icmd="bcftools view -G -Oz normalized_mac1_age60cutoff_updateids.vcf -o vep_input_mac1_age60cutoff_updateids.vcf.gz" \
--instance-type mem2_ssd1_v2_x32 \
--brief \
--destination "${projectid}:${1}/04_annotated"

job-J4y4kFjJJFG986BBzx7bbpFV


In [ ]:
%%bash -s $OUTPUT_DIR

# index vcf

dx run swiss-army-knife \
-iin="${1}/04_annotated/vep_input_mac1_age60cutoff_updateids.vcf.gz" \
-icmd="bcftools index vep_input_mac1_age60cutoff_updateids.vcf.gz" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--destination "${projectid}:${1}/04_annotated/"


In [ ]:
# Download vep input file
! dx download {OUTPUT_DIR}/04_annotated/vep_input_mac1_age60cutoff_updateids.vcf.gz --overwrite
! dx download {OUTPUT_DIR}/04_annotated/vep_input_mac1_age60cutoff_updateids.vcf.gz.csi --overwrite

# copy into vep data folder
! cp vep_input_mac1_age60cutoff_updateids.vcf.gz $HOME/vep_data/
! cp vep_input_mac1_age60cutoff_updateids.vcf.gz.csi $HOME/vep_data/


! gunzip $HOME/vep_data/vep_input_mac1_age60cutoff_updateids.vcf.gz
! mv vep_input_mac1_age60cutoff_updateids.vcf $HOME/vep_data/

## Make covariate file

In [ ]:
! dx download {OUTPUT_DIR}/cases_age60cutoff.txt --overwrite
! dx download {OUTPUT_DIR}/control_age60cutoff.txt --overwrite

In [10]:
df_cases = pd.read_csv("cases_age60cutoff.txt", sep = "\t")
df_control = pd.read_csv("control_age60cutoff.txt", sep = "\t")

In [11]:
df_cases['AGE'] = df_cases['AGE_OF_RECRUIT']
df_cases['FID'] = df_cases['ID']
df_cases['IID'] = df_cases['ID']

In [12]:
df_control['AGE'] = df_control['AGE_OF_RECRUIT']
df_control['FID'] = df_control['ID']
df_control['IID'] = df_control['ID']

In [13]:
df_cases = df_cases[["FID","IID","GENETIC_SEX","AGE","PC1","PC2","PC3","PC4","PC5", "ANCESTRY"]]
df_control = df_control[["FID","IID","GENETIC_SEX","AGE","PC1","PC2","PC3","PC4","PC5", "ANCESTRY"]]

In [14]:
df_cases_with_pheno = df_cases.copy()
df_control_with_pheno = df_control.copy()

In [15]:
df_cases_with_pheno["PHENO"] = 2
df_control_with_pheno["PHENO"] = 1

In [16]:
df_cases_with_pheno = df_cases_with_pheno[["FID","IID", "PHENO", "GENETIC_SEX","AGE","PC1","PC2","PC3","PC4","PC5", "ANCESTRY"]]
df_control_with_pheno = df_control_with_pheno[["FID","IID", "PHENO", "GENETIC_SEX","AGE","PC1","PC2","PC3","PC4","PC5", "ANCESTRY"]]

In [ ]:
df_covar = pd.concat([df_cases_with_pheno, df_control_with_pheno])
df_covar['GENETIC_SEX'] = df_covar['GENETIC_SEX'].replace({'Male': 1, 'Female': 2})

In [18]:
df_covar.to_csv("covariate_age60cutoff.txt", index=False, sep="\t")

In [ ]:
! dx upload covariate_age60cutoff.txt --path $OUTPUT_DIR/covariate_age60cutoff.txt

In [20]:
df_covar.shape

(199662, 11)

In [ ]:
## Make file of sample IDs to keep 

samples_toKeep = df_covar[['FID', 'IID']].copy()

samples_toKeep.to_csv("samplestoKeep_age60cutoff.txt", sep = '\t', index=False)
! dx upload samplestoKeep_age60cutoff.txt --path {OUTPUT_DIR}/samplestoKeep_age60cutoff.txt

## Create individual covariate files for each ancestry

### Create covariate files for each ancestry

In [ ]:
! dx download {OUTPUT_DIR}/covariate_age60cutoff.txt

In [26]:
covariate = pd.read_csv("covariate_age60cutoff.txt", sep = "\t")
covariate
ancestry_list = covariate["ANCESTRY"].unique().tolist()

# create list for storing case control counts
cohort_stats = []
for ancestry in ancestry_list:
    final_df = covariate[covariate["ANCESTRY"] == ancestry]
    cohort_stats.append((ancestry,(final_df["PHENO"] == 2).sum(),
                         (final_df["PHENO"] == 1).sum(), 
                         f'{final_df.shape[0]}',
                         f'{final_df["AGE"].isna().sum()}', 
                         f'{round(final_df[final_df["PHENO"] == 2]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 2]["AGE"].std(),2)}',
                         f'{round(final_df[final_df["PHENO"] == 1]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 1]["AGE"].std(),2)}',))
    
    # create covariate and pheno files
    final_df.to_csv(f"{ancestry}_covariate.txt", sep="\t", index=False, na_rep="NA")
    ! cut -f 1,2,3 {ancestry}_covariate.txt > {ancestry}_pheno.txt
    
    #create samples to keep file
    ! cut -f 1,2 {ancestry}_covariate.txt > {ancestry}_samplestoKeep.txt
    
    

# Write case control counts to tab-delimited file
with open(f'case_control_cts_wgs.txt', 'w') as f:
    f.write("Ancestry\tCases\tControls\tTotal Samples\tParticipants missing age\tAges (PD Cases)\tAges (Neurologically Healthy Controls)\n")
    for row in cohort_stats:
        f.write(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}\t{row[4]}\t{row[5]}\t{row[6]}\n")


In [ ]:
! dx upload case_control_cts_wgs.txt --path {OUTPUT_DIR}/case_control_cts_wgs.txt

## Make individual PLINK files for each ancestry 

In [21]:
# make directories for storing ancestry results

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "mkdir", "-p", f"{OUTPUT_DIR}/{ancestry}/"],
        check=True
    )


In [ ]:
# make a vcf file for each ancestry

projectid = "project"
DATA_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/03_normalized"
BASE_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"
  

    # Build the PLINK command as a single string
    plink_cmd = (
        f"plink2 "
        f"--pfile {ancestry}_{GENE}.coding_nonsyn "
        f"--extract {GENE}.all_variants_toKeep.txt "
        f"--max-alleles 2 "
        f"--recode vcf id-paste=iid "
        f"--mac 1 "
        f"--out {ancestry}_{GENE}.coding_nonsyn"
    )

    # Build dx run command
    cmd = [
        "dx", "run", "swiss-army-knife",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.pgen",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.psam",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.pvar",
        f"-iin={BASE_DIR}/{GENE}.all_variants_toKeep.txt",
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    # Run the command

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)




In [ ]:
# make a bfile set for each ancestry

projectid = "project"
DATA_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/03_normalized"
BASE_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"
  

    # Build the PLINK command as a single string
    plink_cmd = (
        f"plink2 "
        f"--pfile {ancestry}_{GENE}.coding_nonsyn "
        f"--make-bed "
        f"--out {ancestry}_{GENE}.coding_nonsyn"
    )

    # Build dx run command
    cmd = [
        "dx", "run", "swiss-army-knife",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.pgen",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.psam",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.pvar",
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--name", f"{ancestry} make bed file",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    # Run the command

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)




In [13]:
# download covariate file
! dx download {OUTPUT_DIR}/covariate_age60cutoff.txt

[===========================================================>] Completed 24,427,388 of 24,427,388 bytes (100%) /opt/notebooks/covariate_age60cutoff.txtt


In [ ]:
## Split VCF into multiancestry files
%%bash -s $OUTPUT_DIR

### remove sites with MAC <1

dx run swiss-army-knife \
-iin="${1}/03_normalized/normalized_age60cutoff.vcf" \
-iin="/data/hg38.fa" \
-icmd="bcftools view -i 'MAC>=1' normalized_age60cutoff.vcf -O v -o normalized_mac1_age60cutoff.vcf" \
--brief \
--instance-type mem2_ssd1_v2_x64 \
--destination "${projectid}:${1}/03_normalized/"

In [60]:
# Upload ancestries samples to keep

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "upload", "-p", f"{ancestry}_samplestoKeep.txt", "--path", f"{OUTPUT_DIR}/{ancestry}/"],
        check=True
    )

In [61]:
# Upload ancestries covariate files

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "upload", f"{ancestry}_covariate.txt", "--path", f"{OUTPUT_DIR}/{ancestry}/"],
        check=True
    )

In [62]:
# Upload ancestries pheno files
for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "upload", f"{ancestry}_pheno.txt", "--path", f"{OUTPUT_DIR}/{ancestry}/"],
        check=True
    )

In [64]:
# Upload vep files

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "upload", "UKB_vep_output_no_pick.txt", "vep_download_instructions.txt", "--path", f"{OUTPUT_DIR}/{ancestry}/"],
        check=True
    )

## Perform Annotation with VEP

In [16]:
# Enter these lines into the command line
 
# docker pull ensemblorg/ensembl-vep

# mkdir vep_data

#mkdir vep_data/cache 
# cd cache
# curl -O https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_refseq_vep_115_GRCh38.tar.gz
# tar -xzf homo_sapiens_vep_115_GRCh38.tar.gz


# mkdir vep_data/fasta
# cd fasta
#curl -O https://ftp.ensembl.org/pub/release-115/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
# gzip -d Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
# bgzip Homo_sapiens.GRCh38.dna.primary_assembly.fa

In [ ]:
# Download vep input file and move to vep folder
! dx download {OUTPUT_DIR}/04_annotated/vep_input_mac1_age60cutoff_updateids.vcf -o /home/dnanexus/vep_data/ --overwrite

In [5]:
# run container to generate VEP output
import os
import subprocess
vep_cmd = [
    "docker", "run",
    "-v", f"{os.environ['HOME']}/vep_data:/data",
    "ensemblorg/ensembl-vep",
    "vep",
    "--cache",
    "--offline",
    "--force_overwrite",
    "--input_file", f"./vep_input_mac1_age60cutoff_updateids.vcf.gz",
    "--fork", "4",
    "--output_file", f"./UKB_vep_output.txt",
    "--flag_pick_allele",
    "--refseq",
    "--no_stats",
    "--hgvs", # get hgvs annotations 
    "--use_transcript_ref",
    "--assembly", "GRCh38",
    "--fasta", "/opt/vep/.vep/homo_sapiens/115_GRCh38/Homo_sapiens.GRCh38.dna.toplevel.fa.gz",
    "--force_overwrite"
]
result = subprocess.run(vep_cmd)

if result.returncode == 0:
    print("VEP annotation complete!")
else:
    print("VEP failed :(")

VEP annotation complete!


In [ ]:
! dx upload $HOME/vep_data/UKB_vep_output.txt {OUTPUT_DIR}/

In [8]:
# move result into notebooks folder

! cp $HOME/vep_data/UKB_vep_output.txt /opt/notebooks/UKB_vep_output.txt 

In [9]:
# Remove header from vep output, subset to important columns and move to work directory

! grep -v "##" /opt/notebooks/UKB_vep_output.txt | cut -f 1-7,14 > /opt/notebooks/UKB_vep_output_noheader.txt  

In [10]:
# Create another file where we filter to variants flagged by PICK

! awk 'NR==1 || /PICK/' UKB_vep_output_noheader.txt > UKB_vep_output_pick_filtered.txt

In [13]:
# read in file from VEP filtered output

gene = pd.read_csv(f"/opt/notebooks/UKB_vep_output_pick_filtered.txt", sep="\t", dtype="str")

In [14]:
# ref seq ID for ITSN1 is 6453, filter for it to keep only ITSN1 variants
gene_subset = gene[gene["Gene"] == "6453"].copy()

In [15]:
# edit dataframe to create variants to keep file that matches plink format
gene_subset["Location"] = gene_subset["Location"].str.replace("21:", "")
variation_df = gene_subset["#Uploaded_variation"].str.split(":", expand=True)
location_df = gene_subset["Location"].str.split("-", expand=True)


location_df.columns = ["Start", "End"]
location_df["End"] = location_df.apply(lambda row: row["Start"] if row["End"] is None else row["End"], axis=1)

variation_df.columns = ["Chr", "Start", "Ref", "Alt"]
variation_df.drop(columns=["Start"], inplace=True)

fixed_columns = pd.concat([variation_df, location_df], axis=1)
gene_df = pd.concat([fixed_columns,gene_subset], axis=1)
hgvs_annotations = gene_df["Extra"].str.split(":", expand=True).iloc[:,2]
gene_df["HGVS"] = hgvs_annotations

# drop unnecessary columns
gene_df.drop(columns=["Location", "Feature", "Feature_type", "Extra"], axis=1, inplace=True)

gene_df["Gene"] = "ITSN1"
gene_df.rename(columns={"#Uploaded_variation":"SNP"}, inplace=True)

In [17]:
gene_df

,Chr,Ref,Alt,Start,End,SNP,Allele,Gene,Consequence,HGVS
420,chr21,CAG,C,33641742,33641743,chr21:33641741:CAG:C,-,ITSN1,upstream_gene_variant,None
421,chr21,C,CA,33641741,33641742,chr21:33641741:C:CA,A,ITSN1,upstream_gene_variant,None
422,chr21,G,A,33641743,33641743,chr21:33641743:G:A,A,ITSN1,upstream_gene_variant,None
423,chr21,G,C,33641745,33641745,chr21:33641745:G:C,C,ITSN1,upstream_gene_variant,None
424,chr21,C,A,33641748,33641748,chr21:33641748:C:A,A,ITSN1,upstream_gene_variant,None
...,...,...,...,...,...,...,...,...,...,...
66972,chr21,A,G,33903421,33903421,chr21:33903421:A:G,G,ITSN1,downstream_gene_variant,None
66973,chr21,C,G,33903426,33903426,chr21:33903426:C:G,G,ITSN1,downstream_gene_variant,None
66974,chr21,G,C,33903437,33903437,chr21:33903437:G:C,C,ITSN1,downstream_gene_variant,None
66975,chr21,A,C,33903444,33903444,chr21:33903444:A:C,C,ITSN1,downstream_gene_variant,None


In [19]:
# write value_counts output to file
gene_df["Consequence"].value_counts().to_csv(f"{GENE}_all_vep_consequences.txt", sep="\t")

In [20]:
vep_coding_variants = gene_df[gene_df["Consequence"].str.contains('^splice_donor_variant|^splice_acceptor_variant|^missense|^stop_gained|frame|^protein_altering|transcript_ablation|transcript_amplification', regex=True)].copy()
vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                                                                             568
missense_variant,splice_region_variant                                                        20
frameshift_variant                                                                            19
stop_gained                                                                                   16
splice_donor_variant                                                                          11
inframe_deletion                                                                               8
inframe_insertion                                                                              2
splice_acceptor_variant                                                                        2
frameshift_variant,splice_region_variant                                                       1
stop_gained,frameshift_variant                                                                 1
protein_altering_v

In [21]:
# filter out any synonymous variants
final_vep_coding_variants = vep_coding_variants[~(vep_coding_variants["Consequence"].str.contains('synonymous_variant', regex=False))].copy()
final_vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                                                                             568
missense_variant,splice_region_variant                                                        20
frameshift_variant                                                                            19
stop_gained                                                                                   16
splice_donor_variant                                                                          11
inframe_deletion                                                                               8
inframe_insertion                                                                              2
splice_acceptor_variant                                                                        2
frameshift_variant,splice_region_variant                                                       1
stop_gained,frameshift_variant                                                                 1
protein_altering_v

In [22]:
# Save ids in PLINK format
final_vep_coding_variants["SNP"].to_csv(f"{GENE}.all_variants_toKeep.txt", sep="\t",  index=False, header=False)

In [23]:
# save consequences annotation file for later
final_vep_coding_variants.to_csv(f"{GENE}_vep_coding_variant_annotations.txt",  sep="\t",  index=False)

In [ ]:
# upload final files to dnanexus
! dx upload {GENE}_all_vep_consequences.txt --path {OUTPUT_DIR}/
! dx upload {GENE}_vep_coding_variant_annotations.txt --path {OUTPUT_DIR}/
! dx upload {GENE}.all_variants_toKeep.txt --path {OUTPUT_DIR}/
! dx upload UKB_vep_output_pick_filtered.txt --path {OUTPUT_DIR}/

## Burden Analyses using RVTests


In [ ]:
# bgzip each vcf

projectid = "project"
DATA_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/03_normalized"
BASE_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"

    cmd = [
        "dx", "run", "swiss-army-knife",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf",
        f"-icmd=bgzip -k {ancestry}_{GENE}.coding_nonsyn.vcf",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDERR:\n", e.stderr)


In [ ]:
# tabix each vcf

projectid = "project"
DATA_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/03_normalized"
BASE_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"

    cmd = [
        "dx", "run", "swiss-army-knife",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz",
        f"-icmd=tabix -f -p vcf {ancestry}_{GENE}.coding_nonsyn.vcf.gz",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


In [ ]:
projectid = "project"
DATA_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/03_normalized"
BASE_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"

    cmd = [
        "dx", "run", "swiss-army-knife",
        f"-iin={OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz",
        f"-icmd=tabix -f -p vcf {ancestry}_{GENE}.coding_nonsyn.vcf.gz",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)

## RV Tests

In [3]:
# If starting from fresh vm, dx download necessary files for rv tests
# Get vcf files for rvtests
for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"
    subprocess.run(
        ["dx", "download", f"{OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz*"],
        check=True
    )
    
    subprocess.run(
        ["dx", "download", f"{OUTPUT_DIR}/{ancestry}_rvtests_covariate_file.txt"],
        check=True
    )
    
    subprocess.run(
        ["dx", "download", f"{OUTPUT_DIR}/{ancestry}_covariate.txt"],
        check=True
    )

In [ ]:
# create cov file that matches rvtest pheno file header
for ancestry in ancestry_list:
    rv_df = pd.read_csv(f"{ancestry}_covariate.txt", sep = "\t")
    rv_df = rv_df.rename(columns={"GENETIC_SEX":"SEX"})
    rv_df.columns = rv_df.columns.str.lower()
    rv_df["fatid"] = 0
    rv_df["matid"] = 0
    rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age"]]
    rv_df.to_csv(f'{ancestry}_rvtests_covariate_file.txt', sep='\t', index=False, header=True)



In [ ]:
# Upload rv tests covariate files
for ancestry in ancestry_list:
    OUTPUT_DIR = f"results/11625_ITSN1_WGS_Pan_Ancestry/{ancestry}"
    subprocess.run(
        ["dx", "upload", f"{ancestry}_rvtests_covariate_file.txt", "--path", f"{OUTPUT_DIR}/"],
        check=True
    )

In [4]:
# get hg 38 refFlat file from ucsc
! wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz

--2025-12-31 19:15:43--  https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8043021 (7.7M) [application/x-gzip]
Saving to: ‘refFlat.txt.gz’

refFlat.txt.gz      100%[===================>]   7.67M  5.98MB/s    in 1.3s    

2025-12-31 19:15:45 (5.98 MB/s) - ‘refFlat.txt.gz’ saved [8043021/8043021]



In [ ]:
## RVtests with covariates 

for ancestry in ancestry_list:
    cmd = [
        "executable/rvtest",
        "--noweb",
        "--hide-covar",
        "--out", f"{ancestry}_{GENE}.burden",
        "--kernel", "skat,skato",
        "--numThread", "72",
        "--inVcf", f"{ancestry}_{GENE}.coding_nonsyn.vcf.gz",
        "--pheno", f"{ancestry}_rvtests_covariate_file.txt",
        "--pheno-name", "pheno",
        "--gene", "ITSN1",
        "--geneFile", "refFlat.txt.gz",
        "--covar", f"{ancestry}_covariate.txt",
        "--covar-name", "GENETIC_SEX,PC1,PC2,PC3,PC4,PC5"
    ]
    
    subprocess.run(cmd, check=True)

In [ ]:
%%bash

GENE="ITSN1_WGS"
ancestry_list=("FIN" "MDE" "SAS")

for ancestry in "${ancestry_list[@]}"; do
    echo "Running RVTEST for ancestry: ${ancestry}"

    executable/rvtest \
        --noweb \
        --hide-covar \
        --out "${ancestry}_${GENE}.burden" \
        --kernel skat,skato \
        --numThread 72 \
        --inVcf "${ancestry}_${GENE}.coding_nonsyn.vcf.gz" \
        --pheno "${ancestry}_rvtests_covariate_file.txt" \
        --pheno-name pheno \
        --gene ITSN1 \
        --geneFile refFlat.txt.gz \
        --covar "${ancestry}_covariate.txt" \
        --covar-name GENETIC_SEX,PC1,PC2,PC3,PC4,PC5
done


In [ ]:
%%bash

GENE="ITSN1_WGS"
ancestry_list=("EUR")

for ancestry in "${ancestry_list[@]}"; do
    echo "Running RVTEST for ancestry: ${ancestry}"

    executable/rvtest \
        --noweb \
        --hide-covar \
        --out "${ancestry}_${GENE}.burden" \
        --kernel skat,skato \
        --numThread 96 \
        --inVcf "${ancestry}_${GENE}.coding_nonsyn.vcf.gz" \
        --pheno "${ancestry}_rvtests_covariate_file.txt" \
        --pheno-name pheno \
        --gene ITSN1 \
        --geneFile refFlat.txt.gz \
        --covar "${ancestry}_covariate.txt" \
        --covar-name GENETIC_SEX,PC1,PC2,PC3,PC4,PC5
done

In [ ]:
# Upload Skat and SkatO files
ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR","FIN", "MDE", "SAS"]
GENE="ITSN1_WGS"
OUTPUT_DIR = f"/results/11625_ITSN1_WGS_Pan_Ancestry"
for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "upload", f"{ancestry}_{GENE}.burden.log", "--path", f"{OUTPUT_DIR}/{ancestry}/"]
    )
    
    subprocess.run(
        ["dx", "upload", f"{ancestry}_{GENE}.burden.Skat.assoc", "--path", f"{OUTPUT_DIR}/{ancestry}/"]
    )
    
    subprocess.run(
        ["dx", "upload", f"{ancestry}_{GENE}.burden.SkatO.assoc", "--path", f"{OUTPUT_DIR}/{ancestry}/"]
    )
    
    

## Case/Control Frequencies

In [9]:
# download pheno files
# skip if already done
ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "FIN", "MDE", "SAS"]

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}/{ancestry}_pheno.txt"]
    )


In [38]:
# download skat/skato files
# skip if already done
OUTPUT_DIR = f"/results/11625_ITSN1_WGS_Pan_Ancestry"
ancestry_list = ["AAC", "AFR", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "FIN", "MDE", "SAS"]

for ancestry in ancestry_list:
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}/{ancestry}_{GENE}.burden.Skat.assoc"]
    )
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}/{ancestry}_{GENE}.burden.SkatO.assoc"]
    )


### Run PLINK 2 GLM

In [ ]:

projectid = "project"
BASE_DIR  = "results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"{BASE_DIR}/{ancestry}"
    PFILE_PREFIX = f"{ancestry}_{GENE}.coding_nonsyn"

    # Build the plink2 command as a single string (executed inside swiss-army-knife)
    plink_cmd = (
        f"plink2 "
        f"--pfile {PFILE_PREFIX} "
        f"--adjust "
        f"--ci 0.95 "
        f"--pheno-name PHENO "
        f"--covar {ancestry}_covariate.txt "
        f"--covar-name GENETIC_SEX,PC1,PC2,PC3,PC4,PC5 "
        f"--covar-variance-standardize "
        f"--glm hide-covar omit-ref firth-fallback "
        f"--silent "
        f"cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err "
        f"--out {ancestry}_{GENE}"
    )

    # dx run swiss-army-knife wrapper
    cmd = [
        "dx", "run", "swiss-army-knife",
        # PLINK pfile inputs (must match {WORK_DIR}/{PFILE_PREFIX} used in --pfile)
        f"-iin={OUTPUT_DIR}/{PFILE_PREFIX}.pgen",
        f"-iin={OUTPUT_DIR}/{PFILE_PREFIX}.psam",
        f"-iin={OUTPUT_DIR}/{PFILE_PREFIX}.pvar",

        # Analysis inputs referenced by plink_cmd
        f"-iin={OUTPUT_DIR}/{ancestry}_covariate.txt",

        # Command to run in the container
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--name", f"{ancestry} PLINK GLM",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


### Run basic association test

In [ ]:

projectid = "project"
BASE_DIR  = "results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"{BASE_DIR}/{ancestry}"
    BFILE_PREFIX = f"{ancestry}_{GENE}.coding_nonsyn"

    # Build the plink2 command as a single string (executed inside swiss-army-knife)
    plink_cmd = (
        f"plink "
        f"--bfile {BFILE_PREFIX} "
        f"--assoc fisher "
        f"--ci 0.95 "
        f"--silent "
        f"--out {ancestry}_{GENE}"
    )

    # dx run swiss-army-knife wrapper
    cmd = [
        "dx", "run", "swiss-army-knife",
        # PLINK bfile inputs 
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bed",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bim",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.fam",
        # Command to run in the container
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--name", f"{ancestry} PLINK fisher",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


### Get variant frequencies

In [ ]:

projectid = "project"
BASE_DIR  = "results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"{BASE_DIR}/{ancestry}"
    BFILE_PREFIX = f"{ancestry}_{GENE}.coding_nonsyn"

    # Build the plink2 command as a single string (executed inside swiss-army-knife)
    plink_cmd = (
        f"plink "
        f"--bfile {BFILE_PREFIX} "
        f"--freq "
        f"--out {ancestry}_{GENE}"
    )

    # dx run swiss-army-knife wrapper
    cmd = [
        "dx", "run", "swiss-army-knife",
        # PLINK bfile inputs 
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bed",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bim",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.fam",
        # Command to run in the container
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--name", f"{ancestry} PLINK freq",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


### Get recode file

In [ ]:

projectid = "project"
BASE_DIR  = "results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"{BASE_DIR}/{ancestry}"
    BFILE_PREFIX = f"{ancestry}_{GENE}.coding_nonsyn"

    # Build the plink2 command as a single string (executed inside swiss-army-knife)
    plink_cmd = (
        f"plink "
        f"--bfile {BFILE_PREFIX} "
        f"--recode A "
        f"--out {ancestry}_{GENE}"
    )

    # dx run swiss-army-knife wrapper
    cmd = [
        "dx", "run", "swiss-army-knife",
        # PLINK bfile inputs 
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bed",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.bim",
        f"-iin={OUTPUT_DIR}/{BFILE_PREFIX}.fam",
        # Command to run in the container
        f"-icmd={plink_cmd}",
        "--instance-type", "mem2_ssd1_v2_x64",
        "--brief",
        "--name", f"{ancestry} PLINK recode",
        "--destination", f"{projectid}:{OUTPUT_DIR}/"
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("dx run failed")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)


In [ ]:
# dx download the necessary files
BASE_DIR  = "results/11625_ITSN1_WGS_Pan_Ancestry"
GENE = "ITSN1_WGS"

for ancestry in ancestry_list:
    OUTPUT_DIR = f"{BASE_DIR}/{ancestry}"
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}_{GENE}.assoc.fisher"]
    )
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}_{GENE}.raw"]
    )
    
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}_{GENE}.frq"]
    )
    
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.PHENO.glm.logistic.hybrid"]
    )
    subprocess.run(
        ["dx", "download", "--no-progress", "--overwrite", f"{OUTPUT_DIR}/{ancestry}_{GENE}.coding_nonsyn.PHENO.glm.logistic.hybrid.adjusted"]
    )

# download variant annotations
subprocess.run(["dx", "download", "--no-progress", "--overwrite", f"{BASE_DIR}/{GENE}_vep_coding_variant_annotations.txt"])
   

In [ ]:
# merge output files
for ANC in ancestry_list:
    ASSOC_FILE = f"{ANC}_{GENE}.assoc.fisher"
    RECODE_FILE = f"{ANC}_{GENE}.raw"
    GLM_HYBRID_FILE = f"{ANC}_{GENE}.coding_nonsyn.PHENO.glm.logistic.hybrid"
    GLM_ADJUST_FILE = f"{ANC}_{GENE}.coding_nonsyn.PHENO.glm.logistic.hybrid.adjusted"
    FREQ_FILE = f"{ANC}_{GENE}.frq"

    log_hybrid = pd.read_csv(GLM_HYBRID_FILE, sep=r"\s+", usecols=["ID", "A1", "A1_FREQ",
                                                                  "P", "OR", "LOG(OR)_SE", "L95", "U95",
                                                                  "FIRTH?", "ERRCODE"]
    )
    
    log_hybrid.rename(columns={"ID": "SNP"}, inplace=True)

    assoc_adjusted = pd.read_csv(GLM_ADJUST_FILE, sep=r"\s+",
        usecols=["ID", "BONF"])
    
    assoc_adjusted.rename(columns={"ID": "SNP"}, inplace=True)

    assoc = pd.read_csv(ASSOC_FILE, sep=r"\s+",
        usecols=["SNP", "P", "F_A", "F_U"])
    
    assoc.rename(columns={"P": "P_fisher"}, inplace=True)

    freq = pd.read_csv(FREQ_FILE,sep=r"\s+",usecols=["SNP", "MAF"])

    df = pd.merge(log_hybrid, assoc_adjusted, on="SNP", how="right")
    df = pd.merge(df, freq, on="SNP", how="left")

    # merge freq with df
    freq_assoc = pd.merge(assoc, df, on="SNP", how="left")

    # read in recode file
    recode = pd.read_csv(RECODE_FILE, sep=r"\s+")

    # Pre-filter the dataset
    cases_data = recode[recode["PHENOTYPE"] == 2]
    controls_data = recode[recode["PHENOTYPE"] == 1]

    # Make a list from the column names
    column_names = recode.columns.tolist()

    # Drop the first 6 columns to keep the variants
    variants = column_names[6:]

    results = []
    for variant in variants:
        # For cases
        hom_cases = cases_data[cases_data[variant] == 2].shape[0]
        het_cases = cases_data[cases_data[variant] == 1].shape[0]
        total_cases = cases_data.shape[0]
        # For controls
        hom_controls = controls_data[controls_data[variant] == 2].shape[0]
        het_controls = controls_data[controls_data[variant] == 1].shape[0]
        total_controls = controls_data.shape[0]
        results.append({
            "Variant": variant,
            "Hom Cases": hom_cases,
            "Het Cases": het_cases,
            "Total Cases": total_cases,
            "Hom Controls": hom_controls,
            "Het Controls": het_controls,
            "Total Controls": total_controls,
        })

    # Return results
    df_results = pd.DataFrame(results)
    df_results["SNP"] = df_results["Variant"].apply(lambda x: x.rsplit("_", 1)[0])
    df_results = df_results.drop("Variant", axis=1)

    # Merge with assoc results
    full_results = pd.merge(freq_assoc, df_results, on="SNP", how="left")

    # append variant annotation
    annotations = pd.read_csv(
        f"{GENE}_vep_coding_variant_annotations.txt",
        sep="\t",
        usecols=["Consequence", "SNP", "HGVS"]
    )

    full_results_annotated = pd.merge(
        full_results,
        annotations,
        on="SNP",
        how="left"
    )

    # subset to only columns to keep
    clean_full_results = full_results_annotated[
        [
            "SNP", "Consequence", "HGVS", "A1", "A1_FREQ", "FIRTH?",
            "P", "P_fisher", "BONF", "OR", "LOG(OR)_SE", "L95", "U95",
            "F_A", "F_U", "MAF",
            "Hom Cases", "Het Cases", "Total Cases",
            "Hom Controls", "Het Controls", "Total Controls", "ERRCODE"
        ]
    ].copy()

    clean_full_results.insert(1, "Ancestry", ANC)


    # Look at significant SNPs, if any 
    sig_freq = clean_full_results[clean_full_results['P']<0.05]
    sig_snps = sig_freq['SNP'].tolist()
    sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
    
    # save files to working directory
    clean_full_results.to_csv(f"{ANC}_{GENE}_UKB.fullVariantInformation.txt", sep="\t", index=False)
    sig_df_results.to_csv(f"{ANC}_{GENE}_UKB.SignificantVariantInformation.txt" , sep="\t", index=False)
    

### Make combined files

In [ ]:
import pandas as pd
from glob import glob
from pathlib import Path

ancestries = ['AAC','AJ','AFR','AMR','CAH','CAS','EAS','EUR','FIN','MDE','SAS']
tests = ['wgs']
base = Path("/opt/notebooks/").expanduser()
out = base / "ITSN1_UKB_combined_results.xlsx"

In [ ]:
with pd.ExcelWriter(out) as w:
    wrote = False
    for t in tests:
        # SKAT / SKATO
        skat_dfs, skato_dfs = [], []
        for a in ancestries:
            d = base
            for f in glob(str(d / f"{a}_*.Skat.assoc")):
                df = pd.read_csv(f, sep="\t")
                df.insert(0, "Ancestry", a)
                skat_dfs.append(df)
            for f in glob(str(d / f"{a}_*.SkatO.assoc")):
                df = pd.read_csv(f, sep="\t")
                df.insert(0, "Ancestry", a)
                skato_dfs.append(df)
        if skat_dfs:
            pd.concat(skat_dfs, ignore_index=True).to_excel(w, sheet_name=f"{t.upper()}_Skat", index=False)
            wrote = True
        if skato_dfs:
            pd.concat(skato_dfs, ignore_index=True).to_excel(w, sheet_name=f"{t.upper()}_SkatO", index=False)
            wrote = True

        # VARIANT INFO
        all_dfs, sign_dfs = [], []
        for a in ancestries:
            d = base
            for f in glob(str(d / f"{a}_*.fullVariantInformation.txt")):
                df = pd.read_csv(f, sep="\t")
                all_dfs.append(df)
            for f in glob(str(d / f"{a}_*.SignificantVariantInformation.txt")):
                df = pd.read_csv(f, sep="\t")
                sign_dfs.append(df)
        if all_dfs:
            pd.concat(all_dfs, ignore_index=True).to_excel(w, sheet_name=f"{t.upper()}_FullVar", index=False)
            wrote = True
        if sign_dfs:
            pd.concat(sign_dfs, ignore_index=True).to_excel(w, sheet_name=f"{t.upper()}_SignVar", index=False)
            wrote = True

    if not wrote:
        pd.DataFrame({"Message": ["No matching files found."]}).to_excel(w, sheet_name="Empty", index=False)
        print("Error")

In [ ]:
# upload to dnanexus project folder
! dx upload ITSN1_UKB_combined_results.xlsx --path {OUTPUT_DIR}/